In [ ]:
from pathlib import Path
from collections import Counter
import json, sys, tempfile
PROJECT = Path('/home/ec2-user/SageMaker/election_arm')
BUNDLE = PROJECT / 'source_review_pilot_tdm_v1'
sys.path.insert(0, str(BUNDLE))
import proquest_tdm_review as r
pack = r.load_pack(BUNDLE / 'pack.json')
checks = []
for record in pack['records']:
    found = r.recover(PROJECT, record)
    checks.append({k: v for k, v in found.items() if k not in {'quote', 'article'}})
r.atomic_json(BUNDLE / 'retrieval_check.json', {'pack_id':pack['pack_id'], 'records':checks})
print('SOURCE RECOVERY:', sum(x['recovered'] for x in checks), '/', len(checks))
print('VERBATIM QUOTE LOCATIONS:', dict(Counter(x.get('quote_occurrences', 'unavailable') for x in checks)))
for x in checks:
    print(x['audit_id'], x['status'], x['reason'] if not x['recovered'] else 'export hash and article ID/date checked')
with tempfile.TemporaryDirectory(prefix='review-synthetic-check-') as d:
    sample = {'pack_id':'SYNTHETIC_ONLY', 'rubric':'test', 'records':[{'audit_id':'SYNTHETIC_ONLY'}]}
    store = r.ReviewStore(d, sample)
    store.record('SYNTHETIC_ONLY', 'first_pass', {'reviewer':'SYNTHETIC_TEST', 'answers':{'mapping_status':'unavailable', 'blinding':'uncertain', 'initial_note':'Synthetic save check'}})
    reopened = r.ReviewStore(d, sample)
    assert reopened.events('SYNTHETIC_ONLY', 'first_pass') == store.events('SYNTHETIC_ONLY', 'first_pass')
    try:
        reopened.record('SYNTHETIC_ONLY','draft',{})
    except ValueError:
        print('SYNTHETIC SAVE / REOPEN / LOCK: PASS')
    else:
        raise AssertionError('Locked first pass could be overwritten')
print('REAL HUMAN EVENTS:', len(r.ReviewStore(BUNDLE / 'responses', pack).data['events']))
print('Checks complete. Return to ProQuest_Review_Offline.ipynb to review.')
